In [1]:
!pip install openmeteo-requests requests-cache retry-requests pandas numpy scikit-learn matplotlib seaborn
!pip install xgboost tensorflow

In [182]:
# !pip install openmeteo-requests requests-cache retry-requests pandas numpy scikit-learn matplotlib seaborn
# Optional: XGBoost and TensorFlow (comment out if not needed or if you already have them)
# !pip install xgboost tensorflow

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import requests_cache
from retry_requests import retry
import openmeteo_requests

from datetime import date, timedelta

from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

import warnings
warnings.filterwarnings("ignore")


# 1) Config: Location, date range, class mapping

In [183]:
# --- Location (Colombo) ---
LAT, LON = 6.9355, 79.8487
TZ = "auto"  # auto -> Asia/Colombo

# --- Training horizon (adjust as you like) ---
START_DATE = "2020-01-01"
END_DATE   = "2024-12-31"

# --- Rolling window size (hours) ---
WINDOW_H = 24

# --- Your rainfall-to-class mapping (currently applied to *next-hour* rainfall in mm) ---
# def classify_weather_mm_per_hour(rf_next_hour: float) -> str:
#     r = float(rf_next_hour)
#     if r < 4.0:
#         return 'Very Dry'
#     elif r < 6.6:
#         return 'Dry'
#     elif r <= 8.3:
#         return 'Normal'
#     elif r <= 10.0:
#         return 'Wet'
#     else:
#         return 'Very Wet'


# def classify_weather_mm_per_hour(rf_next_hour: float) -> str:
#     r = float(rf_next_hour)
#     if r < 0.2:
#         return 'No Rain'
#     elif r < 0.5:
#         return 'Light Rain'
#     elif r <= 1.1:
#         return 'Moderate Rain'
#     elif r <= 2.96:
#         return 'Heavy Rain'
#     else:
#         return 'Very Heavy Rain'
#     
# 0.2        0.5        1.10000002 2.9599999

def classify_weather_mm_per_3h(rf_next_hours: float) -> str:
    r = float(rf_next_hours)
    if r < 0.4:
        return "Light Rain"
    elif r < 1.3:
        return "Moderate Rain"
    else:
        return "Heavy Rain"

#0.40000001 1.29999995
# If you later decide to switch to a *daily* next-24h target, replace the target construction step,
# and keep this same function (since it matches daily mm thresholds).


If you want scientifically consistent yet balanced classes:

Start with WMO guidelines for rainfall intensity:

Light: < 2.5 mm/hour

Moderate: 2.5–10 mm/hour

Heavy: 10–50 mm/hour

Very Heavy: > 50 mm/hour

Adjust them for your 3-hour window (multiply by 3 → 7.5, 30, 150 mm).

# 2) Fetch hourly data from Open-Meteo

In [184]:
# Cached + retried session
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.25)
openmeteo = openmeteo_requests.Client(session=retry_session)

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": LAT,
    "longitude": LON,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "hourly": ["temperature_2m", "precipitation"],
    "timezone": TZ
}

responses = openmeteo.weather_api(url, params=params)
response = responses[0]

hourly = response.Hourly()
time_index = pd.date_range(
    start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
    end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
    freq=pd.Timedelta(seconds=hourly.Interval()),
    inclusive="left"
)

df_hourly = pd.DataFrame({
    "time_utc": time_index,
    "temperature_2m": hourly.Variables(0).ValuesAsNumpy(),
    "precipitation": hourly.Variables(1).ValuesAsNumpy()
})

# Convert to Asia/Colombo (when TZ="auto", timestamps are UTC in payload—convert for readability)
df_hourly["time"] = df_hourly["time_utc"].dt.tz_convert("Asia/Colombo")
df_hourly = df_hourly.drop(columns=["time_utc"]).sort_values("time").reset_index(drop=True)




In [185]:
df_hourly

In [186]:
# def build_aggregated_dataset(df: pd.DataFrame, window_h: int = 24):
#     """
#     Returns a frame with features aggregated over the previous `window_h` hours
#     and target = next-hour rainfall class.
#     """
#     recs = []
#     df = df.sort_values("time").reset_index(drop=True)
# 
#     for i in range(window_h, len(df)-1):
#         past = df.iloc[i-window_h:i]
#         now  = df.iloc[i]          # current hour (end of window)
#         future = df.iloc[i+1]      # the hour we want to predict
# 
#         # Aggregated features from past 24h
#         rec = {
#             "time": now["time"],  # reference time (end of input window)
#             "temp_mean_24h": past["temperature_2m"].mean(),
#             "temp_max_24h":  past["temperature_2m"].max(),
#             "temp_min_24h":  past["temperature_2m"].min(),
#             "temp_std_24h":  past["temperature_2m"].std(),
#             "rain_sum_24h":  past["precipitation"].sum(),
#             "rain_max_24h":  past["precipitation"].max(),
#             "rain_hours_24h": int((past["precipitation"] > 0).sum()),
#             "last_temp": now["temperature_2m"],
#             "last_rain": now["precipitation"],
#         }
# 
#         # Target: classify next-hour rainfall amount
#         rec["y_reg_next_hour"] = float(future["precipitation"])
#         rec["y_cls_next_hour"] = classify_weather_mm_per_hour(rec["y_reg_next_hour"])
#         recs.append(rec)
# 
#     return pd.DataFrame(recs)
# 
# def build_sequential_dataset(df: pd.DataFrame, window_h: int = 24):
#     """
#     Returns X (N, window_h, 2), y_class (N,), y_reg (N,)
#     where features are [temperature_2m, precipitation] sequences over past 24h,
#     target is next-hour rainfall class/regression.
#     """
#     df = df.sort_values("time").reset_index(drop=True)
#     X_seq, y_cls, y_reg, times = [], [], [], []
# 
#     for i in range(window_h, len(df)-1):
#         seq = df.loc[i-window_h:i-1, ["temperature_2m", "precipitation"]].values  # shape: (24, 2)
#         future_rain = float(df.loc[i, "precipitation"])  # next hour relative to end of seq
# 
#         X_seq.append(seq)
#         y_reg.append(future_rain)
#         y_cls.append(classify_weather_mm_per_hour(future_rain))
#         times.append(df.loc[i-1, "time"])  # end time of the input sequence
# 
#     X_seq = np.array(X_seq)
#     y_reg = np.array(y_reg)
#     y_cls = np.array(y_cls)
#     return X_seq, y_cls, y_reg, pd.Series(times, name="time")


# 3) Build rolling 24-hour datasets

In [187]:
def build_aggregated_dataset(df: pd.DataFrame, window_h: int = 24, predict_h: int = 3):
    """
    Returns a DataFrame with features aggregated over the previous `window_h` hours,
    and target = total rainfall over the next `predict_h` hours (classified).
    
    Example:
      window_h = 24 → use past 24 hours as input
      predict_h = 3 → predict total rainfall for next 3 hours (t+1, t+2, t+3)
    """
    recs = []
    df = df.sort_values("time").reset_index(drop=True)

    for i in range(window_h, len(df) - predict_h):
        past = df.iloc[i - window_h:i]           # past 24 hours
        now = df.iloc[i - 1]                     # last hour of input window (hour 24)
        future = df.iloc[i:i + predict_h]        # next 3 hours: [i, i+1, i+2]

        # Aggregated features from past 24 hours
        rec = {
            "time": now["time"],  # reference time = end of input window
            "temp_mean_24h": past["temperature_2m"].mean(),
            "temp_max_24h": past["temperature_2m"].max(),
            "temp_min_24h": past["temperature_2m"].min(),
            "temp_std_24h": past["temperature_2m"].std(),
            "rain_sum_24h": past["precipitation"].sum(),
            "rain_max_24h": past["precipitation"].max(),
            "rain_hours_24h": int((past["precipitation"] > 0).sum()),
            "last_temp": now["temperature_2m"],
            "last_rain": now["precipitation"],
        }

        # Target: total rainfall over next `predict_h` hours
        total_future_rain = future["precipitation"].sum()
        rec["y_reg_next_hours"] = float(total_future_rain)
        rec["y_cls_next_hours"] = classify_weather_mm_per_3h(total_future_rain)

        recs.append(rec)

    return pd.DataFrame(recs)


def build_sequential_dataset(df: pd.DataFrame, window_h: int = 24, predict_h: int = 3):
    """
    Builds a sequential dataset for time-series models (LSTM/CNN/Transformer).
    
    Each sample:
      - Input:  past `window_h` hours of [temperature_2m, precipitation]  → shape (window_h, 2)
      - Target: total rainfall over the next `predict_h` hours (numeric + classified)

    Example:
      window_h = 24  → past 24 hours used as input
      predict_h = 3  → predict total rainfall from next 3 hours (t+1, t+2, t+3)
    """
    df = df.sort_values("time").reset_index(drop=True)
    X_seq, y_cls, y_reg, times = [], [], [], []

    # Loop until we have enough future data (len(df) - predict_h ensures no overflow)
    for i in range(window_h, len(df) - predict_h):
        # Sequence of past 24 hours (features)
        seq = df.loc[i - window_h:i - 1, ["temperature_2m", "precipitation"]].values  # shape: (24, 2)

        # Future window (the next predict_h hours)
        future = df.loc[i:i + predict_h - 1, "precipitation"]

        # Target: total rainfall over next predict_h hours
        total_future_rain = float(future.sum())

        # Save sample
        X_seq.append(seq)
        y_reg.append(total_future_rain)
        y_cls.append(classify_weather_mm_per_3h(total_future_rain))
        times.append(df.loc[i - 1, "time"])  # timestamp marking end of input window

    # Convert to arrays
    X_seq = np.array(X_seq)
    y_reg = np.array(y_reg)
    y_cls = np.array(y_cls)

    return X_seq, y_cls, y_reg, pd.Series(times, name="time")


In [188]:
PREDICT_H = 3  # next 3 hours rainfall total

df_agg = build_aggregated_dataset(df_hourly, window_h=WINDOW_H, predict_h=PREDICT_H)
X_seq, y_seq_cls, y_seq_reg, seq_times = build_sequential_dataset(df_hourly, window_h=WINDOW_H, predict_h=PREDICT_H)

df_agg.head(), X_seq.shape, len(y_seq_cls)

In [189]:
# Check how many samples (rows) belong to each rainfall class
class_counts = df_agg["y_cls_next_hours"].value_counts()
print(class_counts)

# If you want it as percentages:
class_percentages = df_agg["y_cls_next_hours"].value_counts(normalize=True) * 100
print("\nPercentage distribution:")
print(class_percentages.round(2))


In [190]:
import pandas as pd

# Create a Series from y_cls (since it's a NumPy array)
seq_class_counts = pd.Series(y_seq_cls).value_counts()
seq_class_percentages = pd.Series(y_seq_cls).value_counts(normalize=True) * 100

print("Sequential Dataset Class Counts:")
print(seq_class_counts)
print("\nPercentage distribution:")
print(seq_class_percentages.round(2))


In [191]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.histplot(df_agg["y_reg_next_hours"], bins=50, kde=True)
plt.xlabel("3-hour total rainfall (mm)")
plt.ylabel("Frequency")
plt.title("Distribution of 3-hour rainfall totals")
plt.show()


In [192]:
import numpy as np

# Get percentiles (excluding 0 rainfall to avoid dominance)
non_zero = df_agg.loc[df_agg["y_reg_next_hours"] > 0, "y_reg_next_hours"]
# percentiles = np.percentile(non_zero, [20, 40, 60, 80])
percentiles = np.percentile(non_zero, [33.3, 66.6])
print("Suggested thresholds (mm per 3 hours):", percentiles)


### Understanding the Percentile Thresholds

This finds the **20th, 40th, 60th, and 80th percentiles** of all *non-zero* 3-hour rainfall totals.

#### Meaning:
- **20%** of all rainy 3-hour periods have ≤ **0.2 mm** rain  
- **40%** have ≤ **0.5 mm**  
- **60%** have ≤ **1.1 mm**  
- **80%** have ≤ **2.96 mm**  
- The remaining **20%** have > **2.96 mm**

#### Interpretation:
You’re dividing the rainy samples into **five roughly equal groups** (*quintiles*),  
each representing a different **rainfall intensity range**.


# 4) Train/Test split (time-aware)

In [193]:
# --- Aggregated ---
split_idx = int(0.8 * len(df_agg))
train_agg, test_agg = df_agg.iloc[:split_idx], df_agg.iloc[split_idx:]

features_agg = [c for c in df_agg.columns if c not in ["time", "y_reg_next_hours", "y_cls_next_hours"]]
X_train_agg = train_agg[features_agg].values
X_test_agg  = test_agg[features_agg].values
y_train_agg = train_agg["y_cls_next_hours"].values
y_test_agg  = test_agg["y_cls_next_hours"].values

# Optional scaling for tree models is not required, but harmless:
scaler_agg = StandardScaler(with_mean=True, with_std=True)
X_train_agg_s = scaler_agg.fit_transform(X_train_agg)
X_test_agg_s  = scaler_agg.transform(X_test_agg)

# --- Sequential ---
N = len(X_seq)
split_idx_seq = int(0.8 * N)
X_train_seq, X_test_seq = X_seq[:split_idx_seq], X_seq[split_idx_seq:]
y_train_seq, y_test_seq = y_seq_cls[:split_idx_seq], y_seq_cls[split_idx_seq:]


In [194]:
from sklearn.preprocessing import LabelEncoder

# Encode categorical labels to numeric
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train_agg)
y_test_enc  = le.transform(y_test_agg)


# 5A) Train Aggregated models (+ simple tuning)

In [195]:
from sklearn.model_selection import GridSearchCV

def evaluate_classifier(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average="macro")
    print(f"{name} — Acc: {acc:.3f}, Macro-F1: {f1:.3f}")
    print(classification_report(y_true, y_pred))
    cm = confusion_matrix(y_true, y_pred, labels=np.unique(y_true))
    disp_labels = list(np.unique(y_true))
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=disp_labels, yticklabels=disp_labels)
    plt.title(f"{name} — Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

# --- RandomForest (with small grid) ---
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_grid = {
    "n_estimators": [80, 200, 400],
    "max_depth": [None, 4, 8, 14],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}
rf_cv = GridSearchCV(
    rf, rf_grid, scoring="f1_macro", cv=3, n_jobs=-1, verbose=0
)
rf_cv.fit(X_train_agg, y_train_agg)
rf_best = rf_cv.best_estimator_
print("RF best params:", rf_cv.best_params_)

y_pred_rf = rf_best.predict(X_test_agg)
evaluate_classifier("RandomForest (Agg)", y_test_agg, y_pred_rf)

# --- ExtraTrees (often strong baseline) ---
et = ExtraTreesClassifier(random_state=42, n_jobs=-1)
et_grid = {
    "n_estimators": [300, 600],
    "max_depth": [None, 10, 16],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}
et_cv = GridSearchCV(
    et, et_grid, scoring="f1_macro", cv=3, n_jobs=-1, verbose=0
)
et_cv.fit(X_train_agg, y_train_agg)
et_best = et_cv.best_estimator_
print("ExtraTrees best params:", et_cv.best_params_)

y_pred_et = et_best.predict(X_test_agg)
evaluate_classifier("ExtraTrees (Agg)", y_test_agg, y_pred_et)

# --- XGBoost (optional) ---
if HAS_XGB:
    xgb = XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        nthread=-1,
        random_state=42,
        tree_method="hist"  # fast
    )
    xgb_grid = {
        "n_estimators": [300, 500],
        "max_depth": [4, 6],
        "learning_rate": [0.05, 0.1],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0]
    }

    xgb_cv = GridSearchCV(
        xgb, xgb_grid, scoring="f1_macro", cv=3, n_jobs=-1, verbose=0
    )
    xgb_cv.fit(X_train_agg, y_train_enc)  # Use encoded labels
    xgb_best = xgb_cv.best_estimator_
    print("XGBoost best params:", xgb_cv.best_params_)

    y_pred_xgb_enc = xgb_best.predict(X_test_agg)
    y_pred_xgb = le.inverse_transform(y_pred_xgb_enc)  # Decode back to text

    evaluate_classifier("XGBoost (Agg)", y_test_agg, y_pred_xgb)
else:
    print("XGBoost not available — skipping.")


# 5B) Train Sequential model (LSTM)

In [196]:
# Encode classes to integers for Keras
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train_seq_i = le.fit_transform(y_train_seq)
y_test_seq_i  = le.transform(y_test_seq)
n_classes = len(le.classes_)
le.classes_


In [197]:
# Build LSTM
import tensorflow as tf
from tensorflow.keras import layers, models

tf.keras.backend.clear_session()

model = models.Sequential([
    layers.Input(shape=(WINDOW_H, 2)),
    layers.LSTM(64, return_sequences=False),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(n_classes, activation="softmax")
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()


In [198]:
# Train/validation split (time-aware)
val_frac = 0.2
cut = int((1 - val_frac) * len(X_train_seq))
X_tr, X_val = X_train_seq[:cut], X_train_seq[cut:]
y_tr, y_val = y_train_seq_i[:cut], y_train_seq_i[cut:]

callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5, min_lr=1e-5),
    tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True, monitor="val_accuracy")
]

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=128,
    callbacks=callbacks,
    verbose=0
)

plt.figure(figsize=(6,4))
plt.plot(history.history["accuracy"], label="train_acc")
plt.plot(history.history["val_accuracy"], label="val_acc")
plt.legend(); plt.title("LSTM Training"); plt.show()

# Evaluate
probs = model.predict(X_test_seq, verbose=0)
y_pred_seq_i = probs.argmax(axis=1)
y_pred_seq = le.inverse_transform(y_pred_seq_i)

evaluate_classifier("LSTM (Seq)", y_test_seq, y_pred_seq)


# 6) Side-by-side comparison

In [199]:
results = []

def collect_result(name, y_true, y_pred):
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro-F1": f1_score(y_true, y_pred, average="macro")
    })

collect_result("RandomForest (Agg)", y_test_agg, y_pred_rf)
collect_result("ExtraTrees (Agg)", y_test_agg, y_pred_et)
if HAS_XGB:
    collect_result("XGBoost (Agg)", y_test_agg, y_pred_xgb)
collect_result("LSTM (Seq)", y_test_seq, y_pred_seq)

pd.DataFrame(results).sort_values("Macro-F1", ascending=False)


# Hybrid LSTM + XGBoost Model

In [200]:
# ============================================================
# 🌧️ Hybrid LSTM + XGBoost Model (auto-safe final version)
# ============================================================

from tensorflow.keras.models import Model
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# Build model and find hidden Dense layer automatically
# ============================================================

# Force the LSTM model to be built if not yet done
if not model.built:
    model.build(input_shape=(None, X_train_seq.shape[1], X_train_seq.shape[2]))

# Preview layers
print("Model layers:")
for i, layer in enumerate(model.layers):
    print(f"{i}: {layer.__class__.__name__} — {layer.name}")

# Automatically find the last Dense layer *before* the output layer
dense_layers = [i for i, l in enumerate(model.layers)
                if isinstance(l, tf.keras.layers.Dense)]
if len(dense_layers) < 2:
    raise ValueError("Expected at least two Dense layers (hidden + output).")

hidden_layer_index = dense_layers[-2]  # second-last Dense layer (ReLU one)
hidden_output = model.layers[hidden_layer_index].output

# Build the feature extractor model safely
feature_extractor = Model(inputs=model.inputs, outputs=hidden_output)

# Extract hidden features
X_train_features = feature_extractor.predict(X_train_seq, verbose=0)
X_test_features  = feature_extractor.predict(X_test_seq, verbose=0)

print("\nExtracted LSTM feature shapes:")
print("Train:", X_train_features.shape, " Test:", X_test_features.shape)

# ============================================================
# Train XGBoost on the extracted features
# ============================================================

xgb_hybrid = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    nthread=-1,
    random_state=42,
    tree_method="hist"
)

xgb_hybrid.fit(X_train_features, y_train_seq_i)

y_pred_hybrid_i = xgb_hybrid.predict(X_test_features)
y_pred_hybrid = le.inverse_transform(y_pred_hybrid_i)

# ============================================================
# Evaluate Hybrid Model
# ============================================================

def evaluate_classifier(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average="macro")
    print(f"{name} — Acc: {acc:.3f}, Macro-F1: {f1:.3f}")
    print(classification_report(y_true, y_pred))
    cm = confusion_matrix(y_true, y_pred, labels=np.unique(y_true))
    disp_labels = list(np.unique(y_true))
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=disp_labels, yticklabels=disp_labels)
    plt.title(f"{name} — Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

evaluate_classifier("Hybrid LSTM + XGBoost", y_test_seq, y_pred_hybrid)


# Fine-tune Window Size & Prediction Horizon

In [201]:
window_sizes = [12, 24, 36]
predict_horizons = [1, 3, 6]

results = []

for w in window_sizes:
    for p in predict_horizons:
        df_agg = build_aggregated_dataset(df_hourly, window_h=w, predict_h=p)
        X = df_agg.drop(columns=["time", "y_reg_next_hours", "y_cls_next_hours"])
        y = df_agg["y_cls_next_hours"]
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

        rf = RandomForestClassifier(n_estimators=80, max_depth= 4, random_state=0)
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        f1  = f1_score(y_test, y_pred, average="macro")

        results.append((w, p, acc, f1))

results_df = pd.DataFrame(results, columns=["window_h", "predict_h", "Accuracy", "Macro-F1"])
display(results_df)
